In [1]:
"""
Cell 1: INSTALL DEPENDENCY
"""

!pip install -q -U accelerate bitsandbytes pytorch-crf seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.7 MB/s eta 0:00:00


In [2]:
"""
Cell 2: IMPORT
"""
from huggingface_hub import login
import ast
import random
import warnings

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from dataclasses import dataclass

from tqdm.auto import tqdm

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)

from torchcrf import CRF

from transformers import (
    AutoTokenizer,
    AutoModel,
    BitsAndBytesConfig
)

warnings.filterwarnings("ignore")

In [3]:
"""
Cell 3: CONFIG
"""
@dataclass
class Config:
    train_path: str = "/kaggle/input/datasets/quanhh42/phusroyalvihos/train.csv"
    val_path: str = "/kaggle/input/datasets/quanhh42/phusroyalvihos/dev.csv"
    test_path: str = "/kaggle/input/datasets/quanhh42/phusroyalvihos/test.csv"
    
    model_name: str = "Qwen/Qwen3-8B"
    # model_name: str = "ura-hcmut/ura-llama-2.1-8b"
    
    max_length: int = 256
    batch_size: int = 8
    lr: float = 2e-4
    epochs: int = 40
    ratio: float = 1.0
    focal_gamma: float = 2.0
    alpha: float = 0.75
    seed: int = 42
    patience: int = 7
    hf_token: str = ""
cfg = Config()

In [4]:
class HuggingFaceAuthenticator:
    @staticmethod
    def authenticate(token: str):
        login(token=token)
        print("HuggingFace login successful.")

HuggingFaceAuthenticator.authenticate(cfg.hf_token)

HuggingFace login successful.


In [5]:
"""
Cell 4: SEED
"""
class SeedManager:
    @staticmethod
    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
SeedManager.set_seed(cfg.seed)

In [6]:
"""
Cell 5: DATASET
"""
class ViHOSDatasetLoader:
    def __init__(self, cfg):
        self.cfg = cfg
    def load_csv(self, path):
        df = pd.read_csv(path)
        df = df[["content", "index_spans"]]
        df["content"] = df["content"].astype(str)
        df["index_spans"] = df["index_spans"].apply(ast.literal_eval)
        if self.cfg.ratio < 1.0:
            df = df.sample(
                frac=self.cfg.ratio,
                random_state=self.cfg.seed
            ).reset_index(drop=True)
        return df
    def load(self):
        train_df = self.load_csv(
            self.cfg.train_path
        )
        val_df = self.load_csv(
            self.cfg.val_path
        )
        test_df = self.load_csv(
            self.cfg.test_path
        )
        return (
            train_df,
            val_df,
            test_df
        )

In [7]:
"""
Cell 6: TOKENIZER
"""
class TokenizerManager:
    def __init__(self, cfg):
        self.cfg = cfg
    def load(self):
        tokenizer = AutoTokenizer.from_pretrained(
            self.cfg.model_name,
            trust_remote_code=True
        )
        tokenizer.pad_token = tokenizer.eos_token
        tokenizer.padding_side = "right"
        return tokenizer
tokenizer = TokenizerManager(cfg).load()

config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

In [8]:
"""
Cell 7: ALIGNER
"""
LABEL2ID = {
    "O": 0,
    "HATE": 1
}
ID2LABEL = {
    0: "O",
    1: "HATE"
}
class HateSpanAligner:
    def __init__(
        self,
        tokenizer,
        cfg
    ):
        self.tokenizer = tokenizer
        self.cfg = cfg
    def build_char_binary(
        self,
        text,
        spans
    ):
        binary = [0] * len(text)
        for idx in spans:
            if 0 <= idx < len(text):
                binary[idx] = 1
        return binary
    def char_to_token_labels(
        self,
        offsets,
        char_binary
    ):
        labels = []
        for start, end in offsets:
            if start == end:
                labels.append(-100)
                continue
            token_hate = 0
            for idx in range(start, end):
                if idx < len(char_binary):
                    if char_binary[idx] == 1:
                        token_hate = 1
                        break
            labels.append(token_hate)
        return labels
    def align(
        self,
        text,
        spans
    ):
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.cfg.max_length,
            return_offsets_mapping=True
        )
        char_binary = self.build_char_binary(
            text,
            spans
        )
        token_labels = self.char_to_token_labels(
            encoding["offset_mapping"],
            char_binary
        )
        return {
            "input_ids": encoding["input_ids"],
            "attention_mask": encoding["attention_mask"],
            "offset_mapping": encoding["offset_mapping"],
            "token_labels": token_labels,
            "char_binary": char_binary
        }

aligner = HateSpanAligner(
    tokenizer,
    cfg
)

In [9]:
"""
Cell 8: DATASET
"""
class HiddenStateDataset(Dataset):
    def __init__(
        self,
        hidden_states,
        labels,
        texts,
        offsets,
        char_binary
    ):
        self.hidden_states = hidden_states
        self.labels = labels
        self.texts = texts
        self.offsets = offsets
        self.char_binary = char_binary
    def __len__(self):
        return len(self.hidden_states)
    def __getitem__(self, idx):
        return {
            "hidden_states": self.hidden_states[idx],
            "labels": self.labels[idx],
            "text": self.texts[idx],
            "offsets": self.offsets[idx],
            "char_binary": self.char_binary[idx]
        }

In [10]:
"""
Cell 9: COLLATOR
"""
class Collator:
    def __call__(self, batch):
        max_len = max(
            x["hidden_states"].shape[0]
            for x in batch
        )
        hidden_states = []
        labels = []
        masks = []
        texts = []
        offsets = []
        char_binary = []
        for item in batch:
            seq_len = item["hidden_states"].shape[0]
            pad_len = max_len - seq_len
            hidden = F.pad(
                item["hidden_states"],
                (0, 0, 0, pad_len)
            )
            label = item["labels"] + [-100] * pad_len
            mask = [1] * seq_len + [0] * pad_len
            offs = item["offsets"] + [(0, 0)] * pad_len
            hidden_states.append(hidden)
            labels.append(label)
            masks.append(mask)
            texts.append(item["text"])
            offsets.append(offs)
            char_binary.append(
                item["char_binary"]
            )
        return {
            "hidden_states": torch.stack(hidden_states),
            "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(masks),
            "texts": texts,
            "offsets": offsets,
            "char_binary": char_binary
        }

collator = Collator()

In [11]:
"""
Cell 10: FEATURE EXTRACTOR
"""
class QwenFeatureExtractor:
    def __init__(
        self,
        cfg,
        tokenizer,
        aligner
    ):
        self.cfg = cfg
        self.tokenizer = tokenizer
        self.aligner = aligner
        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )
        self.model = self.load_model()
    def load_model(self):
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        )
        model = AutoModel.from_pretrained(
            self.cfg.model_name,
            trust_remote_code=True,
            quantization_config=bnb_config,
            device_map={"": 0},
            low_cpu_mem_usage=True
        )
        model.eval()
        model.config.use_cache = False
        for param in model.parameters():
            param.requires_grad = False
        return model
    def extract_batch(
        self,
        texts,
        spans_list
    ):
        encodings = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.cfg.max_length,
            padding=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        input_ids = encodings["input_ids"].to(
            self.device,
            non_blocking=True
        )
        attention_mask = encodings[
            "attention_mask"
        ].to(
            self.device,
            non_blocking=True
        )
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            batch_hidden = (
                outputs.last_hidden_state
                .detach()
                .cpu()
                .float()
            )
        del outputs
        torch.cuda.empty_cache()
        hidden_states = []
        labels = []
        offsets = []
        char_binary = []
        for i in range(len(texts)):
            text = texts[i]
            spans = spans_list[i]
            offset_mapping = encodings[
                "offset_mapping"
            ][i].tolist()
            seq_len = int(
                attention_mask[i]
                .sum()
                .item()
            )
            offset_mapping = offset_mapping[:seq_len]
            hidden = batch_hidden[i][:seq_len]
            char_bin = self.aligner.build_char_binary(
                text,
                spans
            )
            token_labels = self.aligner.char_to_token_labels(
                offset_mapping,
                char_bin
            )
            hidden_states.append(hidden)
            labels.append(token_labels)
            offsets.append(offset_mapping)
            char_binary.append(char_bin)
        del input_ids
        del attention_mask
        del batch_hidden
        del encodings
        torch.cuda.empty_cache()
        return (
            hidden_states,
            labels,
            offsets,
            char_binary
        )
    def extract(self, dataframe):
        hidden_states = []
        labels = []
        texts = []
        offsets = []
        char_binary = []
        batch_size = self.cfg.batch_size
        for start in tqdm(
            range(0, len(dataframe), batch_size)
        ):
            batch_df = dataframe.iloc[
                start:start + batch_size
            ]
            batch_texts = batch_df[
                "content"
            ].tolist()
            batch_spans = batch_df[
                "index_spans"
            ].tolist()
            (
                batch_hidden,
                batch_labels,
                batch_offsets,
                batch_binary
            ) = self.extract_batch(
                batch_texts,
                batch_spans
            )
            hidden_states.extend(batch_hidden)
            labels.extend(batch_labels)
            texts.extend(batch_texts)
            offsets.extend(batch_offsets)
            char_binary.extend(batch_binary)
        torch.cuda.empty_cache()
        return HiddenStateDataset(
            hidden_states,
            labels,
            texts,
            offsets,
            char_binary
        )
    def cleanup(self):
        del self.model
        torch.cuda.empty_cache()

In [12]:
"""
Cell 11: LOAD DATA
"""
dataset_loader = ViHOSDatasetLoader(cfg)
train_df, val_df, test_df = dataset_loader.load()
extractor = QwenFeatureExtractor(
    cfg,
    tokenizer,
    aligner
)
train_dataset = extractor.extract(
    train_df
)
val_dataset = extractor.extract(
    val_df
)
test_dataset = extractor.extract(
    test_df
)

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen3Model LOAD REPORT from: Qwen/Qwen3-8B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0%|          | 0/1106 [00:00<?, ?it/s]

  0%|          | 0/139 [00:00<?, ?it/s]

  0%|          | 0/139 [00:00<?, ?it/s]

In [13]:
"""
Cell 12: DATALOADER
"""
train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.batch_size,
    shuffle=True,
    collate_fn=collator
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collator
)
test_loader = DataLoader(
    test_dataset,
    batch_size=cfg.batch_size,
    shuffle=False,
    collate_fn=collator
)

In [14]:
"""
Cell 13: FOCAL LOSS
"""
class WeightedFocalLoss(nn.Module):
    def __init__(
        self,
        gamma,
        alpha
    ):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
    def forward(
        self,
        logits,
        labels
    ):
        logits = logits.view(
            -1,
            logits.shape[-1]
        )
        labels = labels.view(-1)
        valid_mask = labels != -100
        logits = logits[valid_mask]
        labels = labels[valid_mask]
        ce_loss = F.cross_entropy(
            logits,
            labels,
            reduction="none"
        )
        pt = torch.exp(-ce_loss)
        alpha_t = torch.where(
            labels == 1,
            self.alpha,
            1 - self.alpha
        )
        focal_loss = (
            alpha_t
            * ((1 - pt) ** self.gamma)
            * ce_loss
        )
        return focal_loss.mean()

In [15]:
"""
Cell 14: HEAD
"""
class TokenClassificationHead(nn.Module):
    def __init__(
        self,
        hidden_size,
        gamma,
        alpha
    ):
        super().__init__()
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(
            hidden_size,
            2
        )
        self.crf = CRF(
            num_tags=2,
            batch_first=True
        )
        self.focal = WeightedFocalLoss(
            gamma,
            alpha
        )
    def forward(
        self,
        hidden_states,
        attention_mask,
        labels=None
    ):
        hidden_states = hidden_states.float()
        hidden_states = self.dropout(
            hidden_states
        )
        logits = self.classifier(
            hidden_states
        )
        if labels is not None:
            crf_labels = labels.clone()
            crf_labels[
                crf_labels == -100
            ] = 0
            focal_loss = self.focal(
                logits,
                labels
            )
            crf_loss = -self.crf(
                logits,
                crf_labels,
                mask=attention_mask.bool(),
                reduction="mean"
            )
            loss = focal_loss + crf_loss
            predictions = self.crf.decode(
                logits,
                mask=attention_mask.bool()
            )
            return {
                "loss": loss,
                "predictions": predictions
            }
        predictions = self.crf.decode(
            logits,
            mask=attention_mask.bool()
        )
        return {
            "predictions": predictions
        }

In [16]:
"""
Cell 15: TRAINER
"""
class Trainer:
    def __init__(
        self,
        model,
        optimizer,
        device
    ):
        self.model = model
        self.optimizer = optimizer
        self.device = device
    def train_epoch(self, loader):
        self.model.train()
        total_loss = 0
        for batch in tqdm(loader):
            hidden_states = batch[
                "hidden_states"
            ].to(self.device)
            
            labels = batch[
                "labels"
            ].to(self.device)
            
            attention_mask = batch[
                "attention_mask"
            ].to(self.device)
            
            outputs = self.model(
                hidden_states=hidden_states,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs["loss"]
            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item()
        return total_loss / len(loader)

In [17]:
"""
Cell 15.5: EARLY STOPPING
"""
class EarlyStopping:
    def __init__(
        self,
        patience=7,
        mode="max"
    ):
        self.patience = patience
        self.mode = mode
        self.best_score = None
        self.counter = 0
        self.should_stop = False

    def step(
        self,
        score
    ):
        if self.best_score is None:
            self.best_score = score
            return
        improved = False
        if self.mode == "max":
            improved = score > self.best_score
        else:
            improved = score < self.best_score
        if improved:
            self.best_score = score
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

In [18]:
"""
Cell 15.6: VALIDATOR
"""
class Validator:
    def __init__(
        self,
        model,
        device
    ):
        self.model = model
        self.device = device
    def evaluate(
        self,
        loader
    ):
        self.model.eval()
        y_true = []
        y_pred = []
        with torch.no_grad():
            for batch in loader:
                hidden_states = batch[
                    "hidden_states"
                ].to(self.device)

                attention_mask = batch[
                    "attention_mask"
                ].to(self.device)

                labels = batch[
                    "labels"
                ].to(self.device)

                outputs = self.model(
                    hidden_states=hidden_states,
                    attention_mask=attention_mask
                )

                predictions = outputs[
                    "predictions"
                ]

                for i in range(len(predictions)):

                    pred_seq = predictions[i]

                    gold_seq = labels[i].tolist()

                    mask_len = int(
                        attention_mask[i]
                        .sum()
                        .item()
                    )

                    pred_seq = pred_seq[:mask_len]

                    gold_seq = gold_seq[:mask_len]

                    for g, p in zip(
                        gold_seq,
                        pred_seq
                    ):

                        if g == -100:

                            continue

                        y_true.append(g)

                        y_pred.append(p)

        return f1_score(
            y_true,
            y_pred,
            average="macro"
        )

In [19]:
"""
Cell 16: INIT MODEL
"""
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

head_model = TokenClassificationHead(
    hidden_size=train_dataset[0]["hidden_states"].shape[-1],
    gamma=cfg.focal_gamma,
    alpha=cfg.alpha
).to(device)
optimizer = torch.optim.AdamW(
    head_model.parameters(),
    lr=cfg.lr
)
trainer = Trainer(
    head_model,
    optimizer,
    device
)

In [20]:
"""
Cell 17: TRAIN
"""
validator = Validator(
    head_model,
    device
)
early_stopping = EarlyStopping(
    patience=cfg.patience,
    mode="max"
)
best_f1 = 0
for epoch in range(cfg.epochs):
    train_loss = trainer.train_epoch(
        train_loader
    )
    val_f1 = validator.evaluate(
        val_loader
    )
    print(
        f"Epoch {epoch+1}"
        f" | Loss: {train_loss:.4f}"
        f" | Val F1: {val_f1:.4f}"
    )
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(
            head_model.state_dict(),
            "best_head.pt"
        )
    early_stopping.step(
        val_f1
    )
    if early_stopping.should_stop:
        print("Early stopping")
        break
head_model.load_state_dict(
    torch.load("best_head.pt")
)

  0%|          | 0/1106 [00:00<?, ?it/s]

Epoch 1 | Loss: 7.1503 | Val F1: 0.7106


  0%|          | 0/1106 [00:00<?, ?it/s]

Epoch 2 | Loss: 5.7425 | Val F1: 0.7531


  0%|          | 0/1106 [00:00<?, ?it/s]

...

  0%|          | 0/1106 [00:00<?, ?it/s]

Epoch 13 | Loss: 3.9199 | Val F1: 0.7536


  0%|          | 0/1106 [00:00<?, ?it/s]

Epoch 14 | Loss: 3.9947 | Val F1: 0.7479
Early stopping


<All keys matched successfully>

In [21]:
"""
Cell 18: EVALUATOR
"""
class Evaluator:
    def __init__(
        self,
        model,
        device
    ):
        self.model = model
        self.device = device
    def token_to_char_binary(
        self,
        predictions,
        offsets,
        text_length
    ):
        binary = [0] * text_length
        for pred, (start, end) in zip(
            predictions,
            offsets
        ):
            if pred == 1:
                for idx in range(start, end):
                    if idx < text_length:
                        binary[idx] = 1
        return binary
    def binary_to_spans(
        self,
        binary
    ):
        return [
            idx
            for idx, value
            in enumerate(binary)
            if value == 1
        ]

    def span_f1(
        self,
        gold_spans,
        pred_spans
    ):
        gold_set = set(gold_spans)
        pred_set = set(pred_spans)
        if len(gold_set) == 0 and len(pred_set) == 0:
            return 1.0
        if len(pred_set) == 0:
            return 0.0
        tp = len(
            gold_set & pred_set
        )
        precision = tp / (
            len(pred_set) + 1e-8
        )

        recall = tp / (
            len(gold_set) + 1e-8
        )

        if precision + recall == 0:

            return 0.0

        return (
            2 * precision * recall
        ) / (
            precision + recall
        )

    def evaluate(self, loader):

        self.model.eval()

        y_true = []

        y_pred = []

        ranked_samples = []

        with torch.no_grad():

            for batch in tqdm(loader):

                hidden_states = batch[
                    "hidden_states"
                ].to(self.device)

                attention_mask = batch[
                    "attention_mask"
                ].to(self.device)

                outputs = self.model(
                    hidden_states=hidden_states,
                    attention_mask=attention_mask
                )

                predictions = outputs[
                    "predictions"
                ]

                labels = batch["labels"]

                for i in range(len(predictions)):

                    pred_seq = predictions[i]

                    gold_seq = labels[i].tolist()

                    mask_len = int(
                        attention_mask[i]
                        .sum()
                        .item()
                    )

                    pred_seq = pred_seq[:mask_len]

                    gold_seq = gold_seq[:mask_len]

                    filtered_gold = []

                    filtered_pred = []

                    for g, p in zip(
                        gold_seq,
                        pred_seq
                    ):

                        if g == -100:

                            continue

                        filtered_gold.append(g)

                        filtered_pred.append(p)

                    y_true.extend(
                        filtered_gold
                    )

                    y_pred.extend(
                        filtered_pred
                    )

                    pred_binary = self.token_to_char_binary(
                        filtered_pred,
                        batch["offsets"][i][:len(filtered_pred)],
                        len(batch["texts"][i])
                    )

                    gold_binary = batch[
                        "char_binary"
                    ][i]

                    pred_spans = self.binary_to_spans(
                        pred_binary
                    )

                    gold_spans = self.binary_to_spans(
                        gold_binary
                    )

                    overlap = len(
                        set(gold_spans)
                        &
                        set(pred_spans)
                    )

                    score = self.span_f1(
                        gold_spans,
                        pred_spans
                    )

                    ranked_samples.append({

                        "text": batch["texts"][i],

                        "label_spans": gold_spans,

                        "predict_spans": pred_spans,

                        "label_binary": gold_binary,

                        "predict_binary": pred_binary,

                        "overlap": overlap,

                        "span_f1": score
                    })

        ranked_samples = sorted(
            ranked_samples,
            key=lambda x: (
                x["overlap"],
                x["span_f1"],
                len(x["predict_spans"])
            ),
            reverse=True
        )

        return (
            y_true,
            y_pred,
            ranked_samples
        )

In [22]:
"""
Cell 19: TEST
"""

evaluator = Evaluator(
    head_model,
    device
)

y_true, y_pred, ranked_samples = evaluator.evaluate(
    test_loader
)

print("=" * 80)
print("TOKEN ACCURACY")
print(
    accuracy_score(
        y_true,
        y_pred
    )
)

print("=" * 80)
print("TOKEN MACRO F1")
print(
    f1_score(
        y_true,
        y_pred,
        average="macro"
    )
)

print("=" * 80)
print("TOKEN WEIGHTED F1")
print(
    f1_score(
        y_true,
        y_pred,
        average="weighted"
    )
)

print("=" * 80)
print("CLASSIFICATION REPORT")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "O",
            "HATE"
        ],
        digits=6
    )
)

  0%|          | 0/139 [00:00<?, ?it/s]

TOKEN ACCURACY
0.8572938157298847
TOKEN MACRO F1
0.7725362313435956
TOKEN WEIGHTED F1
0.8574335529063523
CLASSIFICATION REPORT
              precision    recall  f1-score   support

           O   0.911956  0.910817  0.911386     16012
        HATE   0.632054  0.635328  0.633686      3861

    accuracy                       0.857294     19873
   macro avg   0.772005  0.773072  0.772536     19873
weighted avg   0.857576  0.857294  0.857434     19873



In [23]:
"""
Cell 20: TOP 20 OVERLAP
"""

for i, sample in enumerate(
    ranked_samples[:20]
):

    print("=" * 100)

    print(f"SAMPLE {i+1}")

    print()

    print("TEXT:")
    print(sample["text"])

    print()

    print("OVERLAP:")
    print(sample["overlap"])

    print()

    print("SPAN F1:")
    print(sample["span_f1"])

    print()

    print("LABEL SPANS:")
    print(sample["label_spans"])

    print()

    print("PREDICT SPANS:")
    print(sample["predict_spans"])

    print()

    print("LABEL BINARY:")
    print(sample["label_binary"])

    print()

    print("PREDICT BINARY:")
    print(sample["predict_binary"])

SAMPLE 1

TEXT:
@Ngoc Quynh Nhu Nguyen Thằng lồn cha với con đĩ mẹ mày không biết dạy mày nên mày mới dựng lồn lên xem vào chuyện của người khác, bộ tao ăn hết của ông nội mày sao mà mày lôi tao vào. Từ đầu tao đâu có lôi tên con đĩ mẹ mày ra nói đâu

OVERLAP:
95

SPAN F1:
0.8675799085965681

LABEL SPANS:
[23, 24, 25, 26, 27, 28, 29, 30, 31, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 70, 71, 72, 78, 79, 80, 86, 87, 88, 89, 90, 91, 92, 93, 133, 134, 135, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 167, 168, 169, 175, 176, 177, 191, 192, 193, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222]

PREDICT SPANS:
[25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 